In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statistics as st

In [4]:
df=pd.read_csv('/content/indian_realestate_dataset_1000.csv')

In [5]:
df

,property_id,city,district,property_type,built_up_area_sqft,price_inr,price_per_sqft_inr,bedrooms,bathrooms,age_years,...,annual_rental_yield_pct,price_category,estimated_roi_5yr_pct,listing_date,days_on_market,status,registration_days,is_gated_community,has_maintenance_fee,monthly_maintenance_inr
0,PROP00001,Chennai,Anna Nagar,Apartment,1526,10000000,6553,2,1,4,...,4.20,Premium,46.02,2024-10-17,215,Available,169,True,False,0
1,PROP00002,Mumbai,Bandra,Studio,424,50000000,117924,1,1,6,...,4.61,Luxury,48.07,2025-01-29,283,Available,50,False,False,100000
2,PROP00003,Chennai,Anna Nagar,Apartment,1333,10000000,7501,3,1,27,...,3.71,Premium,43.55,2024-06-27,312,Available,48,True,True,20000
3,PROP00004,Pune,Viman Nagar,Independent House,4121,15000000,3639,3,2,26,...,3.42,Premium,42.11,2024-10-11,336,Rented,84,False,False,0
4,PROP00005,Bangalore,Indiranagar,Villa,4020,30000000,7462,5,3,28,...,3.31,Luxury,41.53,2025-12-02,198,Sold,46,False,False,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,PROP00996,Bangalore,Jayanagar,Villa,6040,30000000,4966,4,2,5,...,4.66,Luxury,48.30,2025-03-11,88,Available,114,False,True,0
996,PROP00997,Delhi,Gurgaon,Penthouse,7007,40000000,5708,5,3,0,...,4.51,Luxury,47.54,2026-04-22,341,Available,32,True,True,80000
997,PROP00998,Bangalore,Koramangala,Apartment,1921,30000000,15616,3,1,19,...,3.31,Luxury,41.57,2025-10-29,86,Available,31,True,False,60000
998,PROP00999,Mumbai,Bandra,Independent House,4170,50000000,11990,4,3,27,...,4.44,Luxury,47.18,2026-04-10,259,Available,178,False,True,100000


In [8]:
# Identify and quantify missing values with isnull().sum().

df.isnull().sum()

,0
property_id,0
city,0
district,0
property_type,0
built_up_area_sqft,0
price_inr,0
price_per_sqft_inr,0
bedrooms,0
bathrooms,0
age_years,0


In [15]:
# Apply imputation strategies: mean, median, mode, forward-fill.

print(np.mean(df['price_inr']))
print(np.median(df['price_inr']))
print(st.mode(df['price_inr']))


25969629.14
26719297.5
10000000


In [18]:
# Detect and remove or cap outliers using IQR method.

# Data quality log
quality_log = []
clean_df = df.copy()

# Select numeric columns
numeric_cols = df.select_dtypes(include=np.number).columns

for col in numeric_cols:
    q1 = df[col].quantile(0.25)
    q3 = clean_df[col].quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

# Count outliers before treatment
    outlier_mask = (
        (clean_df[col] < lower_bound) |
        (clean_df[col] > upper_bound)
    )

    outlier_count = int(outlier_mask.sum())

# Cap outliers
    clean_df[col] = clean_df[col].clip(
        lower=lower_bound,
        upper=upper_bound
    )

    quality_log.append({
        "Column": col,
        "Cleaning Action": "IQR capping",
        "Affected Rows": outlier_count,
        "Lower Bound": lower_bound,
        "Upper Bound": upper_bound,
        "Reason": "Limit extreme values using IQR bounds"
    })

    print(f"\nColumn: {col}")
    print("Q1:", q1)
    print("Q3:", q3)
    print("IQR:", iqr)
    print("Lower bound:", lower_bound)
    print("Upper bound:", upper_bound)
    print("Outliers detected:", outlier_count)



Column: built_up_area_sqft
Q1: 1247.0
Q3: 4997.5
IQR: 3750.5
Lower bound: -4378.75
Upper bound: 10623.25
Outliers detected: 0

Column: price_inr
Q1: 12000000.0
Q3: 40000000.0
IQR: 28000000.0
Lower bound: -30000000.0
Upper bound: 82000000.0
Outliers detected: 0

Column: price_per_sqft_inr
Q1: 3749.25
Q3: 20833.0
IQR: 17083.75
Lower bound: -21876.375
Upper bound: 46458.625
Outliers detected: 130

Column: bedrooms
Q1: 2.0
Q3: 5.0
IQR: 3.0
Lower bound: -2.5
Upper bound: 9.5
Outliers detected: 0

Column: bathrooms
Q1: 1.0
Q3: 3.0
IQR: 2.0
Lower bound: -2.0
Upper bound: 6.0
Outliers detected: 0

Column: age_years
Q1: 7.0
Q3: 23.0
IQR: 16.0
Lower bound: -17.0
Upper bound: 47.0
Outliers detected: 0

Column: amenities_count
Q1: 4.0
Q3: 7.0
IQR: 3.0
Lower bound: -0.5
Upper bound: 11.5
Outliers detected: 0

Column: walkability_score
Q1: 5.0
Q3: 9.0
IQR: 4.0
Lower bound: -1.0
Upper bound: 15.0
Outliers detected: 0

Column: monthly_rental_estimate_inr
Q1: 40175.25
Q3: 126884.5
IQR: 86709.25
Lower 

In [23]:
# Standardise inconsistent string values (casing, whitespace, types).

# Identify text columns
text_cols = clean_df.select_dtypes(include=["object", "string"]).columns

for col in text_cols:
    before = clean_df[col].copy()

# Remove leading/trailing spaces and repeated whitespace
    clean_df[col] = clean_df[col].apply(
        lambda x: " ".join(x.split()) if isinstance(x, str) else x)


# Standardize casing for text categories
    clean_df[col] = clean_df[col].apply(
        lambda x: x.title() if isinstance(x, str) else x)

    changed_count = int(
        (before.fillna("") !=
         clean_df[col].fillna("<NA>")).sum())

    quality_log.append({
        "Column": col,
        "Cleaning Action": "Whitespace and casing standardization",
        "Affected Rows": changed_count,
        "Lower Bound": None,
        "Upper Bound": None,
        "Reason": "Improve text consistency"})

    print(f"{col}: {changed_count} values standardized")

property_id: 0 values standardized
city: 0 values standardized
district: 0 values standardized
property_type: 0 values standardized
facing: 0 values standardized
amenities: 0 values standardized
price_category: 0 values standardized
listing_date: 0 values standardized
status: 0 values standardized


In [38]:
# Example mappings — verify these against your dataset first
corrections = {
    "city": {
        "Chennai": "Mumbai",
        "Bangalore": "Pune"
    },
    "property_type": {
        "Apartmnt": "Apartment",
        "Flatt": "Flat"
    }
}

# Match column names without assuming capitalization
column_lookup = {
    col.lower().strip(): col for col in clean_df.columns}

for column_name, mapping in corrections.items():
    actual_col = column_lookup.get(column_name.lower())

    if actual_col is not None:
        before = clean_df[actual_col].copy()

        clean_df[actual_col] = clean_df[actual_col].replace(mapping)

        changed_count = int(
            (before.fillna("") !=
             clean_df[actual_col].fillna("")).sum())

        quality_log.append({
            "Column": actual_col,
            "Cleaning Action": "Known spelling corrections",
            "Affected Rows": changed_count,
            "Lower Bound": None,
            "Upper Bound": None,
            "Reason": "Correct verified spelling variations"})

        print(f"{actual_col}: {changed_count}")
    else:
        print(f"Column {column_name}")

city: 0
property_type: 0


In [40]:
# Document cleaning decisions in a data quality log.

quality_log_df = pd.DataFrame(quality_log)

display(quality_log_df)

# Save the log
quality_log_df.to_csv("/content/indian_realestate_dataset_1000.csv",index=False)

# Save the cleaned dataset
clean_df.to_csv("/content/indian_realestate_dataset_1000.csv",index=False)

print("Cleaning completed.")
print("Cleaned dataset:", "/content/indian_realestate_dataset_1000.csv")
print("Quality log:", "/content/indian_realestate_dataset_1000.csv")

print("Original shape:", df.shape)
print("Cleaned shape:", clean_df.shape)

print("\nMissing values after cleaning:")
print(clean_df.isnull().sum())

print("\nDuplicate rows after cleaning:")
print(clean_df.duplicated().sum())

print("\nData quality log:")
display(quality_log_df)

,Column,Cleaning Action,Affected Rows,Lower Bound,Upper Bound,Reason
0,built_up_area_sqft,IQR capping,0,-4.378750e+03,1.062325e+04,Limit extreme values using IQR bounds
1,price_inr,IQR capping,0,-3.000000e+07,8.200000e+07,Limit extreme values using IQR bounds
2,price_per_sqft_inr,IQR capping,130,-2.187638e+04,4.645862e+04,Limit extreme values using IQR bounds
3,bedrooms,IQR capping,0,-2.500000e+00,9.500000e+00,Limit extreme values using IQR bounds
4,bathrooms,IQR capping,0,-2.000000e+00,6.000000e+00,Limit extreme values using IQR bounds
...,...,...,...,...,...,...
69,property_type,Known spelling corrections,0,NaN,NaN,Correct verified spelling variations
70,city,Known spelling corrections,0,NaN,NaN,Correct verified spelling variations
71,property_type,Known spelling corrections,0,NaN,NaN,Correct verified spelling variations
72,city,Known spelling corrections,0,NaN,NaN,Correct verified spelling variations


Cleaning completed.
Cleaned dataset: /content/indian_realestate_dataset_1000.csv
Quality log: /content/indian_realestate_dataset_1000.csv
Original shape: (1000, 25)
Cleaned shape: (1000, 25)

Missing values after cleaning:
property_id                    0
city                           0
district                       0
property_type                  0
built_up_area_sqft             0
price_inr                      0
price_per_sqft_inr             0
bedrooms                       0
bathrooms                      0
age_years                      0
facing                         0
amenities                      0
amenities_count                0
walkability_score              0
monthly_rental_estimate_inr    0
annual_rental_yield_pct        0
price_category                 0
estimated_roi_5yr_pct          0
listing_date                   0
days_on_market                 0
status                         0
registration_days              0
is_gated_community             0
has_maintenance_fe

,Column,Cleaning Action,Affected Rows,Lower Bound,Upper Bound,Reason
0,built_up_area_sqft,IQR capping,0,-4.378750e+03,1.062325e+04,Limit extreme values using IQR bounds
1,price_inr,IQR capping,0,-3.000000e+07,8.200000e+07,Limit extreme values using IQR bounds
2,price_per_sqft_inr,IQR capping,130,-2.187638e+04,4.645862e+04,Limit extreme values using IQR bounds
3,bedrooms,IQR capping,0,-2.500000e+00,9.500000e+00,Limit extreme values using IQR bounds
4,bathrooms,IQR capping,0,-2.000000e+00,6.000000e+00,Limit extreme values using IQR bounds
...,...,...,...,...,...,...
69,property_type,Known spelling corrections,0,NaN,NaN,Correct verified spelling variations
70,city,Known spelling corrections,0,NaN,NaN,Correct verified spelling variations
71,property_type,Known spelling corrections,0,NaN,NaN,Correct verified spelling variations
72,city,Known spelling corrections,0,NaN,NaN,Correct verified spelling variations
